In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Microsoft YaHei'
plt.rcParams['axes.unicode_minus' ] = False

In [38]:
from lib.model import Model
from lib.dataset import Data
from lib.dataloader.generators import ChunkedGenerator, UnchunkedGenerator
import os
import importlib
import json
import torch

In [31]:
exp_path = "checkpoint/VIDEOPOSE_h36m_videoOri_FRAME81_LR0.00025_EPOCH256_BATCH128_May_18_2025_20_36_11"
model_cfg_json_path = os.path.join(exp_path, "configs", "model_config.json")
dataset_cfg_json_path = os.path.join(exp_path, "configs", "data_config.json")
ckpt_path = os.path.join(exp_path, "best_epoch.bin")

In [43]:
model_cfg, data_cfg = {},{} 
with open(model_cfg_json_path, 'r') as f, open (dataset_cfg_json_path, 'r') as f1:
    model_cfg = json.load(f) 
    data_cfg = json.load(f1) 
model_cfg, data_cfg

({'MODEL': 'videoOri',
  'NUM_COARSE_ANG': 8,
  'TRAJECTORY_MODEL': True,
  'BONE_COMPARISON': False,
  'ARCHITECTURE': '3,3,3,3',
  'DROPOUT': 0.25,
  'NUM_FRAMES': 81,
  'CAUSAL': False,
  'CHANNELS': 1024,
  'DENSE': False,
  'NUM_KPTS': 17,
  'INPUT_DIM': 2,
  'DISABLE_OPTIMIZATIONS': False,
  'PRETRAIN': ''},
 {'DATASET': 'h36m',
  'WORLD_3D_GT_EVAL': True,
  'KEYPOINTS': 'gt',
  'TRAIN_SUBJECTS': 'S1,S5,S6,S7,S8',
  'TEST_SUBJECTS': 'S9,S11',
  'GT_3D': 'data/h36m/data_3d_h36m.npz',
  'GT_2D': 'data/h36m/data_2d_h36m_gt.npz',
  'CAMERA_PARAM': '',
  'SUBSET': 1,
  'STRIDE': 1,
  'DOWNSAMPLE': 1,
  'ACTIONS': '*',
  'REMOVE_IRRELEVANT_KPTS': False,
  'FRAME_PATH': '/ssd/yzhan/data/benchmark/3D/showroom/20210702/frame/',
  'ORI_ENCODING': True,
  'INTRINSIC_ENCODING': True,
  'RAY_ENCODING': True,
  'ADD_HEIGHT': False})

In [33]:
train_delegator = Model(model_cfg, {"ADD_HEIGHT": False}, is_train=False)
model =train_delegator.get_pos_model(); 
    

In [34]:
checkpoint = torch.load(ckpt_path, map_location=lambda storage, loc: storage)

In [35]:
model.load_state_dict(checkpoint["model_pos"], strict=True)

<All keys matched successfully>

In [45]:
pose_data = Data(data_cfg)
subjects_train = data_cfg["TRAIN_SUBJECTS"].split(",")
subjects_test = data_cfg["TEST_SUBJECTS"].split(",")
action_filter = None if data_cfg["ACTIONS"] == "*" else data_cfg["ACTIONS"].split(",")
cameras_valid, poses_valid, poses_valid_2d, poses_valid_ori = pose_data.fetch_via_subject(
        subjects_test, action_filter
)
kps_left, kps_right = pose_data.get_2d_kpts()
joints_left, joints_right = pose_data.get_3d_joints()


IndexError: index 11 is out of bounds for axis 0 with size 4

In [ ]:

test_generator = ChunkedGenerator(
        1,
        cameras_valid,
        poses_valid,
        poses_valid_2d,
        poses_valid_ori,
        1,
        pad=False,
        causal_shift=False,
        shuffle=False,
        augment=False,
        kps_left=kps_left,
        kps_right=kps_right,
        joints_left=joints_left,
        joints_right=joints_right,
    )